# Figure and table: grain yield and N use efficiency indicators

Computes agronomic efficiency (AE), recovery efficiency of N (REN), physiological
efficiency (PEN) and the N balance from the plot-level harvest data, and compares
the two fertilization strategies.

**Input** `../data/raw/yield.csv`
**Output** `../figures/yield.png` and the statistics reported in the manuscript table


# LOAD LIBRARIES

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np

# LOAD DATA

In [ ]:
# Load Data
df = pd.read_csv('../data/raw/yield.csv')

# Convert tons to kg
for var in ['DM_grain', 'DM_straw']:
    df[var]*= 1000

# Calculate additional metrics from available ones
df['total_DM'] = df['DM_straw'] + df['DM_grain']
df['N_uptake'] = df['straw_N'] + df['grain_N']
df['HI'] = df['DM_grain'] / df['total_DM'] * 100 # Harvest Index (%)
df['NHI'] = df['grain_N'] / df['N_uptake'] * 100 # Nitrogen Harvest Index (%)
df['N_grain_pct'] = df['grain_N'] / df['DM_grain'] * 100 
df['N_straw_pct'] = df['straw_N'] / df['DM_straw'] * 100
df['N_total_pct'] = df['N_uptake'] / df['total_DM'] * 100
df['protein_pct'] = df['N_grain_pct'] * 5.83 # multiplication factor for whole kerned (Jones, 1941)

display(df)

# CALCULATE INDICATORS

In [ ]:
# We isolate just the two treatments to compare
target_treatments = ['Farm', 'VISTA']
df_compare = df[df['treatment'].isin(target_treatments)].copy()

# MANUALLY ADD N INPUT based on the fertilization GIS map (as applied)
n_input_map = {
    # (Treatment, Rep) : kg N/ha
    ('Farm', 1): 152, 
    ('Farm', 2): 149,
    ('Farm', 3): 152,
    ('VISTA', 1): 147,
    ('VISTA', 2): 144,
    ('VISTA', 3): 147
}
df_compare['N_input_in_season'] = df_compare.apply(
    lambda row: n_input_map.get((row['treatment'], row['rep'])), axis=1
)
# Add the first fertilization before sowing (total N of the solid pig manure) for the total N input
# The basal dressing pre-sowing was applied also to zero plots so for efficiency indices we use only the in-season N input
df_compare['N_input_total'] = df_compare['N_input_in_season'] + 92 # using the total N from the solid pig manure

# Calculate efficiency metrics
df_compare['PFPN'] = df_compare['DM_grain'] / df_compare['N_input_total']
# Calculate agronomic efficiency (extra grain produced for every kg of N added)
zero_ref_yield = df[df['treatment'] == 'Zero'].set_index(['side', 'rep'])['DM_grain'] # reference series indexed by BOTH 'side' and 'rep'
df_compare['baseline_yield'] = df_compare.set_index(['side', 'rep']).index.map(zero_ref_yield)
df_compare['AE'] = (df_compare['DM_grain'] - df_compare['baseline_yield']) / df_compare['N_input_in_season']
# Apparent N recovery efficiency in aboveground biomass (%)
zero_ref_Nuptake = df[df['treatment'] == 'Zero'].set_index(['side', 'rep'])['N_uptake']
df_compare['baseline_N_uptake'] = (df_compare.set_index(['side', 'rep']).index.map(zero_ref_Nuptake))
df_compare['REN'] = ((df_compare['N_uptake'] - df_compare['baseline_N_uptake']) / df_compare['N_input_in_season']) * 100
# N surplus (kg N/ha)
df_compare['N_balance'] = (df_compare['N_input_in_season'] + df_compare['baseline_N_uptake']) - df_compare['N_uptake']
# Physiological efficiency of N [PEN] (kg/kg)
df_compare['PEN'] = (df_compare['DM_grain'] - df_compare['baseline_yield']) / (df_compare['N_uptake'] - df_compare['baseline_N_uptake'])

# Define columns to IGNORE
metadata_cols = ['ID_field', 'date_harvest', 'treatment', 'side', 'rep', 'long', 'lat']

# Select all columns that are NOT in the ignore list 
metrics = [col for col in df_compare.columns if col not in metadata_cols]

display(metrics)
display(df_compare)

Quick stats

In [ ]:
print("--- T-Test Results (Farm vs VISTA) ---")

for var in metrics:
    # Get the two groups
    grp_farm = df_compare[df_compare['treatment'] == 'Farm'][var]
    grp_vista = df_compare[df_compare['treatment'] == 'VISTA'][var]
    # Independent t-test (assuming different fields/variances)
    t_stat, p_val = stats.ttest_ind(grp_farm, grp_vista, equal_var=False)

    sig = "**" if p_val < 0.01 else '*' if p_val < 0.05 else "ns"
    print(f"{var:<15} | Farm: {grp_farm.mean():.2f} vs VISTA: {grp_vista.mean():.2f} | p={p_val:.3f} {sig}")

# PLOT

## CONFIGS

In [ ]:
# Ploting settings
from plot_config import COLORS, LABELS

order = ['BAU', 'VRA']
LABELS = {'BAU': LABELS['A'], 'VRA': LABELS['B']}
COLORS = {'BAU': COLORS['A'], 'VRA': COLORS['B']}

# Helper to add p value from t test
def add_stat_annotation(ax, data, x, y, order):
    """
    Calculates T-test and adds p value
    """
    # Get data for the two groups
    group1 = data[data[x] == order[0]][y]
    group2 = data[data[x] == order[1]][y]
    # Calculate P-value
    t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)
    # Position the annotation for p value at the top center of the plot
    ax.text(0.5, 0.93, f'p = {p_val:.3f}', 
            transform=ax.transAxes, 
            ha='center', va='bottom', color='k')
    
# Function to create barplot with mean and 95% CI, and overlay raw data points
def bar_mean_ci(ax, y, ylabel, df):
    # Barplot for mean and 95% CI
    # Plotting this first so the bars are in the background
    sns.barplot(
        data=df, x='treatment', y=y, order=order,
        hue='treatment', palette=COLORS, 
        # errorbar=('ci', 95),   # Calculates 95% Confidence Interval
        # capsize=0.3,           # Adds horizontal caps to the error bars
        # disable errorbars since we add already the points and the sample size is very small (n=3)
        errorbar=None,
        alpha=0.8,          
        ax=ax,
        legend=False
    )
    # Stripplot for the raw data points
    sns.stripplot(
        data=df, x='treatment', y=y, order=order,
        color='black', size=7, jitter=0, 
        alpha=0.5, ax=ax
    )
    # Stats annotation
    add_stat_annotation(ax, df, 'treatment', y, order)
    # Labels and formatting
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')

def scatter_with_reg(ax, data, x, y, xlabel, ylabel, show_legend=False):
    """
    Plots a scatterplot with a light grey overall regression line.
    """
    # Plot the regression line first (in the background)
    sns.regplot(
        data=data, x=x, y=y, 
        scatter=False,          # Don't plot the raw points here
        color='lightgrey',      # Light grey line
        ci=None,                # Set to 95 if you want the confidence interval ribbon
        ax=ax, 
        line_kws={'linestyle': '--', 'zorder': 1}
    )
    # Plot the styled scatter points on top
    sns.scatterplot(
        data=data, x=x, y=y,
        hue='treatment', style='treatment', s=200,
        palette=COLORS, ax=ax, legend=show_legend,
        zorder=2                # Forces points to be rendered above the line
    )
    # Apply labels
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # Handle custom legend safely
    if show_legend:
        handles, current_labels = ax.get_legend_handles_labels()
        new_labels = [LABELS.get(label, label) for label in current_labels]
        ax.legend(handles=handles, labels=new_labels, title=None, frameon=False, loc='best')

## FIG

In [ ]:
df_comp = df_compare.copy()
# Convert kg to tons for better readability in the plots
for var in ['DM_grain', 'DM_straw', 'total_DM']:
    df_comp[var] /= 1000
# Rename treatments for better readability in the plots
df_comp['treatment'] = df_comp['treatment'].replace({'Farm':'BAU', 'VISTA':'VRA'})

# PLOTTING SETTINGS
sns.set_theme(style="ticks", rc={"axes.spines.top": False})
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
})

# Define the Figure
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# Plot A: biomass vs HI
ax = axes[0,0]
scatter_with_reg(
    ax=ax, data=df_comp, 
    x='total_DM', y='HI', 
    xlabel='Crop biomass (t DM ha$^{-1}$)', ylabel='Harvest index (%)', 
    show_legend=True
)
ax.set_ylim(30, 45)

# Plot B: grain yield vs protein
ax = axes[0,1]
scatter_with_reg(
    ax=ax, data=df_comp, 
    x='DM_grain', y='protein_pct', 
    xlabel='Grain yield (t DM ha$^{-1}$)', ylabel='Protein content (%)', 
    show_legend=False
)

# Plot C: N surplus
ax = axes[1,0]
bar_mean_ci(df=df_comp, ax=ax, y='N_balance', ylabel='Apparent N balance (kg N ha$^{-1}$)')
ax.axhline(0, color='0.4', lw=0.8, ls='--', alpha=0.5)
ax.set_ylim(-10, 70)

# Plot D: NUE
ax = axes[1,1]
bar_mean_ci(df=df_comp, ax=ax, y='REN', ylabel='Apparent N recovery (%)')
ax.set_ylim(0, 120)

# Add panel letters
labels = ['(a)', '(b)', '(c)', '(d)']
for lab, ax in zip(labels, axes.flatten()):
    ax.text(0.01, 0.93, lab, transform=ax.transAxes, weight='bold', size=20)

# Move y-axis to the right for right column
for ax in axes[:, 1]:
    ax.yaxis.tick_right()
    ax.yaxis.set_label_position("right")

# Add top spine
for ax in axes.flatten():
    ax.spines['top'].set_visible(True)

plt.tight_layout()
fig.align_ylabels(axes[:, 0])
fig.align_ylabels(axes[:, 1])
plt.savefig('../figures/yield.png', dpi=300, bbox_inches='tight')
plt.show()


# STATS FOR TABLE

In [ ]:
def generate_full_summary_table(df):
    metrics = {
        # --- Yield & Biomass ---
        'DM_grain': 'Grain Yield (t/ha)',
        'DM_straw': 'Straw Biomass (t/ha)',
        'total_DM': 'Total Biomass (t/ha)',
        'HI': 'Harvest Index (%)',
        'baseline_yield': 'Control (Zero) Yield (t/ha)',
        
        # --- Physiology ---
        'ear_density': 'Ear Density (ears/m²)',
        'TGW': 'Thousand Grain Weight (g)',
        
        # --- Nitrogen Concentration/Quality ---
        'protein_pct': 'Protein Content (%)',
        'N_grain_pct': 'Grain N Concentration (%)',
        'N_straw_pct': 'Straw N Concentration (%)',
        'N_total_pct': 'Plant N Concentration (%)',

        # --- Nitrogen Uptake ---
        'grain_N': 'Grain N Uptake (kg N/ha)',
        'straw_N': 'Straw N Uptake (kg N/ha)',
        'N_uptake': 'Total N Uptake (kg N/ha)',
        'NHI': 'Nitrogen Harvest Index (%)',
        
        # --- Efficiency ---
        'N_input_total': 'Total N Input (kg N/ha)',
        'N_input_in_season': 'In-season N Input (kg N/ha)',
        'PFPN': 'Partial Factor Productivity of N (kg/kg)',
        'AE': 'Agronomic Efficiency of N (kg/kg)',
        'N_balance': 'Apparent N balance (kg N/ha)',
        'REN': 'Apparent N recovery efficiency (%)',
        'PEN': 'Physiological efficiency of N'
    }
    
    table_rows = []
    df_sub = df[df['treatment'].isin(['Farm', 'VISTA'])]
    
    def get_stats_row(data, confidence=0.95):
        """Helper to return mean and half-width CI."""
        n = len(data)
        mu = data.mean()
        if n < 2:
            return mu, 0  # Return mean and zero half-width if not enough data
            print("Not enough data to calculate CI for this metric.")
        
        se = stats.sem(data)
        h = se * stats.t.ppf((1 + confidence) / 2., n - 1)
        return mu, h

    for col, label in metrics.items():
        # Clean data
        farm = df_sub[df_sub['treatment'] == 'Farm'][col].dropna()
        vista = df_sub[df_sub['treatment'] == 'VISTA'][col].dropna()
        
        # Calculate stats
        f_mu, f_h = get_stats_row(farm)
        v_mu, v_h = get_stats_row(vista)
        
        # T-test
        t_stat, p_val = stats.ttest_ind(farm, vista, equal_var=False)
        
        if p_val < 0.01: sig = "**"
        elif p_val < 0.05: sig = "*"
        elif p_val < 0.10: sig = "."
        else: sig = "ns"
        
        digits = 1
        table_rows.append({
            'Parameter': label,
            'BAU': f"{f_mu:.{digits}f} ± {f_h:.{digits}f}",
            'VRA': f"{v_mu:.{digits}f} ± {v_h:.{digits}f}",
            'p-value': round(p_val, 3),
            'Sig': sig
        })
        
    return pd.DataFrame(table_rows)

# Generate and display
df = df_compare.copy()
# Convert kg to tons for better readability in the plots
for var in ['DM_grain', 'DM_straw', 'total_DM']:
    df[var] /= 1000
full_table = generate_full_summary_table(df)
print(full_table.to_string(index=False))